# Controlled-long-tail T1 anchors

One-click, fail-closed launcher for the fixed full CONTROLLED T1 ANCHOR TRAINING RECIPE V1. It never runs T2–T6 and never touches historical replay workspaces.

In [ ]:
# 0 — Only edit this cell
CONDITIONS = ["lt10", "lt50", "lt100"]
SEED = 0
RUN_SMOKE_TEST = True
RUN_TRAINING = True
RUN_EVALUATION = True
GPU_BUDGET_HOURS = 9.0
ALLOW_BUDGET_OVERRUN = False  # True starts a full resumable run even when it cannot finish tonight
BENCHMARK_ITERATIONS = 20
DRIVE_ROOT = "/content/drive/MyDrive/OWL"

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "c46ffe193c7f1ab0edc282214d720f08461736f9"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"
EXPECTED_PYTHON = (3, 13)
EXPECTED_TORCH = "2.11.0+cu128"
EXPECTED_TORCHVISION = "0.26.0+cu128"
EXPECTED_CUDA = "12.8"
DINO_SHA256 = "156f8c4166a23dc2951ae811e39d76a06269c565932edf647c0187e65cd7aa7c"
assert CONDITIONS and len(CONDITIONS) == len(set(CONDITIONS))
assert all(value in ("lt10", "lt50", "lt100") for value in CONDITIONS)
assert SEED == 0 and BENCHMARK_ITERATIONS >= 2

In [ ]:
# 1 — Mount Drive, pin repositories, and install the reviewed OWL package
import hashlib
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

assert sys.version_info[:2] == EXPECTED_PYTHON, sys.version
drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
probe = DRIVE / ".controlled_lt_write_probe"
probe.write_text("ok")
assert probe.read_text() == "ok"
probe.unlink()


def checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def capture(command, **kwargs):
    return checked(command, capture_output=True, **kwargs).stdout.strip()


def pinned_checkout(path, repository, commit):
    path = Path(path)
    if not path.exists():
        checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    assert (path / ".git").is_dir()
    actual_origin = capture(["git", "remote", "get-url", "origin"], cwd=path).removesuffix(".git")
    expected_origin = repository.removesuffix(".git")
    assert actual_origin == expected_origin, (actual_origin, expected_origin)
    checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    checked(["git", "reset", "--hard", commit], cwd=path)
    checked(["git", "clean", "-fdx"], cwd=path)
    assert capture(["git", "rev-parse", "HEAD"], cwd=path) == commit
    return path


ROOT = pinned_checkout("/content/owod-active", OWL_REPOSITORY, OWL_COMMIT)
checked(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-q",
        "-e",
        f"{ROOT}[plots]",
    ]
)
sys.path.insert(0, str(ROOT))
from owl import t1_anchor  # noqa: E402

print("OWL PIN PASS:", OWL_COMMIT)

In [ ]:
# 2 — Reuse the proven Python-3.13 PROB dependency and compiled-MSDA bootstrap
PROB = pinned_checkout("/content/PROB", PROB_REPOSITORY, PROB_COMMIT)


def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def available(name):
    return (
        subprocess.run(
            [sys.executable, "-c", f"import {name}"], cwd=PROB, capture_output=True
        ).returncode
        == 0
    )


if version("einops") != "0.5.0" or not available("einops"):
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "einops==0.5.0"])
if version("pycocotools") != "2.0.5" or not available("pycocotools"):
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "Cython==3.1.3"])
    checked(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-build-isolation",
            "--no-deps",
            "--force-reinstall",
            "pycocotools==2.0.5",
        ]
    )
wheels = {
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
    "wandb": "wandb==0.18.7",
}
missing = [spec for module, spec in wheels.items() if not available(module)]
if missing:
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", *missing])
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
if pip_check.returncode and "jedi" in (pip_check.stdout + pip_check.stderr).lower():
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "jedi==0.19.2"])
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], text=True, capture_output=True
    )
if pip_check.returncode:
    print(pip_check.stdout + pip_check.stderr)
    raise RuntimeError("pip check failed")

DINO = PROB / "models/dino_resnet50_pretrain.pth"
if not DINO.is_file():
    partial = DINO.with_suffix(".pth.part")
    if partial.exists():
        partial.unlink()
    checked(
        [
            "curl",
            "--fail",
            "--location",
            "--retry",
            "3",
            "--output",
            str(partial),
            "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth",
        ]
    )
    assert hashlib.sha256(partial.read_bytes()).hexdigest() == DINO_SHA256
    partial.replace(DINO)
assert hashlib.sha256(DINO.read_bytes()).hexdigest() == DINO_SHA256


def msda_probe():
    code = """import json,torch
from models.ops.functions import ms_deform_attn_func as w
from models.ops.modules import ms_deform_attn as d
print(json.dumps({"cuda":torch.cuda.is_available(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"torch":torch.__version__,"torchvision":__import__("torchvision").__version__,"cuda_version":torch.version.cuda,"wrapper":bool(w.MSDA_AVAILABLE),"downstream":bool(d.MSDA_AVAILABLE),"path":getattr(w.MSDA,"__file__",None)}))"""
    result = subprocess.run([sys.executable, "-c", code], cwd=PROB, text=True, capture_output=True)
    return (
        json.loads(result.stdout.splitlines()[-1])
        if result.returncode == 0 and result.stdout
        else {}
    )


MSDA = msda_probe()
if not (MSDA.get("wrapper") and MSDA.get("downstream")):
    checked([sys.executable, "-m", "pip", "install", "-q", "ninja"])
    checked(
        [sys.executable, "-m", "pip", "install", "--no-build-isolation", "."],
        cwd=PROB / "models/ops",
    )
    MSDA = msda_probe()
assert MSDA.get("cuda") and MSDA.get("wrapper") and MSDA.get("downstream"), MSDA
assert (
    MSDA["torch"] == EXPECTED_TORCH
    and MSDA["torchvision"] == EXPECTED_TORCHVISION
    and MSDA["cuda_version"] == EXPECTED_CUDA
), MSDA
assert "T4" in MSDA["gpu"], MSDA

coco_smoke = r"""import numpy as np, torch
if "float" not in np.__dict__: np.float=float
if "NPY_OWNDATA" not in np.__dict__: np.NPY_OWNDATA=4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
c=COCO(); c.dataset={"info":{},"licenses":[],"images":[{"id":1,"width":32,"height":32}],"categories":[{"id":1,"name":"x","supercategory":"x"}],"annotations":[{"id":1,"image_id":1,"category_id":1,"bbox":[4.,5.,10.,11.],"area":110.,"iscrowd":0}]}; c.createIndex(); e=CocoEvaluator(c,("bbox",)); e.update({1:{"boxes":torch.tensor([[4.,5.,14.,16.]]),"scores":torch.tensor([.99]),"labels":torch.tensor([1])}}); e.synchronize_between_processes(); e.accumulate(); assert float(e.coco_eval["bbox"].stats[0])>.99"""
coco_smoke = coco_smoke.replace("e.accumulate(); assert", "e.accumulate(); e.summarize(); assert")
checked([sys.executable, "-c", coco_smoke], cwd=PROB)
print("PROB CUDA/MSDA/COCO BOOTSTRAP PASS:", MSDA)

In [ ]:
# 3 — Materialize the exact requested JPEG union and one shared initialization
WORK_ROOT = DRIVE / "anchors/controlled_lt_v1/seed0"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
JPEG_ROOT = Path("/content/data/controlled_lt/JPEGImages")
INIT = WORK_ROOT / "prob_t1_seed0_init.pth"
checked(
    [
        sys.executable,
        str(ROOT / "tools/materialize_t1_anchor_images.py"),
        "--conditions",
        ",".join(CONDITIONS),
        "--jpeg-root",
        str(JPEG_ROOT),
        "--execute",
    ]
)
if not INIT.is_file():
    environment = os.environ.copy()
    environment["PYTHONHASHSEED"] = str(SEED)
    checked(
        [
            sys.executable,
            str(ROOT / "tools/prepare_t1_anchor_training.py"),
            "create-initialization",
            "--prob-root",
            str(PROB),
            "--output",
            str(INIT),
        ],
        env=environment,
    )
INIT_SHA = hashlib.sha256(INIT.read_bytes()).hexdigest()
INIT_META = json.loads(INIT.with_suffix(".initialization.json").read_text())
assert INIT_META["sha256"] == INIT_SHA and INIT_META["prob_commit"] == PROB_COMMIT
assert (
    INIT_META["torch_version"] == EXPECTED_TORCH
    and INIT_META["torchvision_version"] == EXPECTED_TORCHVISION
    and INIT_META["cuda_version"] == EXPECTED_CUDA
)
print("SHARED INITIALIZATION PASS:", INIT, INIT_SHA, INIT_META["model_state_sha256"])

In [ ]:
# 4 — Deterministically materialize/verify each isolated filtered-XML view
WORKSPACES = {condition: WORK_ROOT / f"t1_anchor__{condition}__seed0" for condition in CONDITIONS}
for condition, workspace in WORKSPACES.items():
    state = t1_anchor.workspace_state(workspace, condition)
    print(condition, "initial state:", state)
    if state == "DONE":
        continue
    if state == "INCOMPLETE NON-RESUMABLE":
        raise RuntimeError(f"{condition}: inspect incomplete workspace {workspace}")
    data_root = Path(f"/content/data/controlled_lt/{condition}/OWOD")
    checked(
        [
            sys.executable,
            str(ROOT / "tools/prepare_t1_anchor_training.py"),
            "preflight",
            "--condition",
            condition,
            "--prob-root",
            str(PROB),
            "--work-root",
            str(WORK_ROOT),
            "--data-root",
            str(data_root),
            "--jpeg-root",
            str(JPEG_ROOT),
            "--initialization",
            str(INIT),
            "--initialization-sha",
            INIT_SHA,
            "--owl-commit",
            OWL_COMMIT,
            "--materialize",
        ]
    )
print("CONTROLLED LT DATA PREFLIGHTS PASS")

In [ ]:
# 5 — Real exact-path CUDA smoke + live seconds/iteration benchmark
SMOKES = {}
for condition, workspace in WORKSPACES.items():
    if t1_anchor.workspace_state(workspace, condition) == "DONE":
        continue
    receipt = workspace / "cuda_training_smoke.json"
    if RUN_SMOKE_TEST and not receipt.is_file():
        checked(
            [
                sys.executable,
                str(ROOT / "tools/train_t1_anchor.py"),
                "--condition",
                condition,
                "--prob-root",
                str(PROB),
                "--workspace",
                str(workspace),
                "--initialization",
                str(INIT),
                "--initialization-sha",
                INIT_SHA,
                "--owl-commit",
                OWL_COMMIT,
                "--benchmark-iterations",
                str(BENCHMARK_ITERATIONS),
                "--smoke-only",
                "--execute",
            ]
        )
    if receipt.is_file():
        SMOKES[condition] = json.loads(receipt.read_text())
    elif RUN_TRAINING:
        raise RuntimeError(f"{condition}: full training forbidden without smoke receipt")
assert not RUN_TRAINING or set(SMOKES) | {
    c for c, w in WORKSPACES.items() if t1_anchor.workspace_state(w, c) == "DONE"
} == set(CONDITIONS)
print("LIVE BENCHMARKS:", {c: s["benchmark"]["seconds_per_iteration"] for c, s in SMOKES.items()})

In [ ]:
# 6 — Budget gate; never shorten or tune the fixed recipe silently
EVAL_HOURS_PER_PASS = (4308 / 1000 * 6.462035541195477 + 0.3) / 60
ETAS = {}
for condition, smoke in SMOKES.items():
    trained = (WORKSPACES[condition] / f"t1_{condition}.pth").is_file()
    training = 0.0 if trained else smoke["estimated_training_hours"][condition]
    evaluation = EVAL_HOURS_PER_PASS if trained else 11 * EVAL_HOURS_PER_PASS
    ETAS[condition] = {
        "training_hours": training,
        "evaluation_hours": evaluation,
        "total_hours": training + evaluation,
    }
requested_hours = sum(row["total_hours"] for row in ETAS.values())
print(
    json.dumps(
        {
            "per_condition": ETAS,
            "requested_total_hours": requested_hours,
            "budget_hours": GPU_BUDGET_HOURS,
        },
        indent=2,
    )
)
BUDGET_BLOCKED = RUN_TRAINING and requested_hours > GPU_BUDGET_HOURS and not ALLOW_BUDGET_OVERRUN
if BUDGET_BLOCKED:
    print(
        "BUDGET GATE: full V1 training was NOT started. Do not call a shortened run a final anchor."
    )

In [ ]:
# 7 — Full one-condition-at-a-time training, exact resume classification, immediate evaluation
if RUN_TRAINING and not BUDGET_BLOCKED:
    for condition, workspace in WORKSPACES.items():
        state = t1_anchor.workspace_state(workspace, condition)
        if state == "DONE":
            continue
        if state not in ("READY", "INCOMPLETE RESUMABLE"):
            raise RuntimeError(f"{condition}: {state}")
        final_checkpoint = workspace / f"t1_{condition}.pth"
        if not final_checkpoint.is_file():
            command = [
                sys.executable,
                str(ROOT / "tools/train_t1_anchor.py"),
                "--condition",
                condition,
                "--prob-root",
                str(PROB),
                "--workspace",
                str(workspace),
                "--initialization",
                str(INIT),
                "--initialization-sha",
                INIT_SHA,
                "--owl-commit",
                OWL_COMMIT,
                "--execute",
            ]
            if state == "INCOMPLETE RESUMABLE":
                command.append("--resume")
            checked(command)
        else:
            print(condition, "validated final checkpoint exists; continuing with evaluation")
        if RUN_EVALUATION:
            checked(
                [
                    sys.executable,
                    str(ROOT / "tools/evaluate_t1_anchor.py"),
                    "--condition",
                    condition,
                    "--prob-root",
                    str(PROB),
                    "--workspace",
                    str(workspace),
                    "--execute",
                ]
            )
        print(condition, "final state:", t1_anchor.workspace_state(workspace, condition))

In [ ]:
# 8 — Combined learnability report only after every requested anchor is validated
STATES = {
    condition: t1_anchor.workspace_state(workspace, condition)
    for condition, workspace in WORKSPACES.items()
}
print("FINAL STATES:", STATES)
if all(value == "DONE" for value in STATES.values()):
    comparison = WORK_ROOT / ("comparison__" + "_".join(CONDITIONS))
    if not (comparison / "anchor_comparison.json").is_file():
        checked(
            [
                sys.executable,
                str(ROOT / "tools/compare_t1_anchors.py"),
                "--work-root",
                str(WORK_ROOT),
                "--conditions",
                ",".join(CONDITIONS),
                "--output",
                str(comparison),
            ]
        )
    print("CONTROLLED LT ANCHOR NOTEBOOK COMPLETE", comparison)
else:
    print(
        "CONTROLLED LT ANCHOR BENCHMARK/RESUME POINT COMPLETE — no incomplete run is labelled a final anchor"
    )